# SVR Inference
2026-02-19 SVR model loading and inference

two different methods to pass the data to the model (just for testing)

In [ ]:
# paths are not correct for the subfolder - check the other scripts for how to set up paths and db access, and adapt as needed

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import json

notebookdir = Path.cwd().parents[2]
sys.path.append(str(notebookdir))  # this way we can import src modules even in different subfolders

notebookdir = Path.cwd().parent
sys.path.append(str(notebookdir))  # this way we can import src modules even in different subfolders

from src.log_utils import log_to_file, log_section

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 250)

In [ ]:
# set directories and filenames, load database
working_dir = Path.cwd().parent.parent

# logging setup
log_file = "8_svr_inference.log"
if Path(log_file).exists():
    Path(log_file).unlink()  # remove existing log file to start fresh
log_to_file("SVR Inference", log_file)
log_section("Model loading", log_file)

In [3]:
# Function to load model and parameters for testing
def load_model_and_params(model_name: Path) -> tuple[Any, dict]:
    """Load trained model and its parameters for inference"""
    model_path = model_name
    params_path = model_path.parent / f"{model_path.stem}_params.json"

    # Load model
    model = joblib.load(model_path)

    # Load parameters
    with open(params_path, "r") as f:
        params_dict = json.load(f)

    print(f"Model loaded from: {model_path}")
    print(f"Parameters loaded from: {params_path}")

    return model, params_dict


# Test loading
model = working_dir / "models" / "SVR_air.joblib"
svr_model, svr_params = load_model_and_params(model_name=model)
print(f"\nLoaded model type: {type(svr_model).__name__}")

print(f"\nModel parameters from JSON:")
for key, value in svr_params.items():
    if key == "feature_columns":
        print(f"  {key}: [{len(value)} features]")
    else:
        print(f"  {key}: {value}")

log_to_file("Model and parameters for Air SVR loaded successfully.", log_file)

Model loaded from: /Users/a/dev/WP3biodegradability/t_half/models/SVR_air.joblib
Parameters loaded from: /Users/a/dev/WP3biodegradability/t_half/models/SVR_air_params.json

Loaded model type: SVR

Model parameters from JSON:
  model_name: SVR
  compartment: air
  target_column: T_half_days
  model_params: {'C': 10, 'epsilon': 0.01, 'gamma': 'scale', 'kernel': 'rbf'}
  feature_columns: [228 features]
  y_value_transformation: log10
  test_size: 0.25
  cross-validation_folds: 5
  model_performance_scores: {'cv_r2_mean': 0.8556248154730095, 'cv_r2_std': 0.06939707549122165, 'cv_mse_mean': 0.10004417292953227, 'cv_mse_std': 0.06426830263423551, 'cv_rmse_log10_mean': 0.31629760183967925, 'cv_rmse_log10_std': 0.25351193785349735, 'cv_mae_mean': 0.1512303637735053, 'cv_mae_std': 0.03972091587815871, 'test_rmse_log10': 0.2401591262477858, 'test_mae_log10': 0.1123428705114175, 'test_rmse_days': 9.345527118693022, 'test_mae_days': 3.181001432554617}
  outlier_removal: True
  preprocessing_drops:

In [4]:
# inference test - create dummy data and run through the same descriptor calculation and scaling steps as training
from src.rdkit_tools import calculate_descriptors
from src.ml_tools import scale_features

# Create test data
df = pd.DataFrame({"Canonical_smiles": ["CCO", "CCN", "CCC"]})
df_with_descriptors = calculate_descriptors(df)

# Check which features are available
print(f"Total features calculated: {len(df_with_descriptors.columns) - 1}")  # -1 for SMILES column
print(f"Model requires: {len(svr_params['feature_columns'])} features")

# Scale features (entire descriptor set)
scaled_features_all = scale_features(df_with_descriptors.drop(columns=["Canonical_smiles"]))

# Check for missing features
required_features = svr_params["feature_columns"]
missing_features = set(required_features) - set(scaled_features_all.columns)
if missing_features:
    print(f"\nWARNING: Missing features: {missing_features}")
else:
    print("\nAll required features are present.")

# Select only the features used in training, in the correct order
# This is critical - must match the exact order used during training
scaled_features = scaled_features_all[required_features].copy()

# Ensure index is reset to avoid any index-related issues
scaled_features = scaled_features.reset_index(drop=True)

print(f"Features prepared for inference: {scaled_features.shape}")
print(f"Feature names match: {list(scaled_features.columns) == required_features}")

# Perform inference - convert to numpy array to avoid feature name checking
predictions_log10 = svr_model.predict(scaled_features.values)

# Convert from log10 to original scale (days)
predictions_days = 10**predictions_log10

# Get uncertainty estimate from CV RMSE (in log10 space)
cv_rmse_log10 = svr_params["model_performance_scores"]["cv_rmse_log10_mean"]

# Calculate uncertainty bounds in original scale (days)
# Log10(T_half) +/- RMSE -> convert to days
lower_bound_log10 = predictions_log10 - cv_rmse_log10
upper_bound_log10 = predictions_log10 + cv_rmse_log10
lower_bound_days = 10**lower_bound_log10
upper_bound_days = 10**upper_bound_log10

# Display results with uncertainty
print("\nInference Results:")
print("=" * 80)
for i, (smiles, pred_log, pred_days, lower, upper) in enumerate(
    zip(df["Canonical_smiles"], predictions_log10, predictions_days, lower_bound_days, upper_bound_days)
):
    uncertainty = (upper - lower) / 2  # Average uncertainty in days
    print(f"Molecule {i + 1}: {smiles}")
    print(f"  Predicted log10(T_half): {pred_log:.4f} ± {cv_rmse_log10:.4f}")
    print(f"  Predicted T_half: {pred_days:.2f} days (95% CI: {lower:.2f} - {upper:.2f} days)")
    print(f"  Uncertainty: ± {uncertainty:.2f} days")
    print()

print(f"Note: Uncertainty based on {svr_params['cross-validation_folds']}-fold CV RMSE = {cv_rmse_log10:.4f} (log10 space)")

Calculating MACCS fingerprints: 100%|██████████| 3/3 [00:00<00:00, 3073.50it/s]

Total features calculated: 375
Model requires: 228 features

All required features are present.
Features prepared for inference: (3, 228)
Feature names match: True

Inference Results:
Molecule 1: CCO
  Predicted log10(T_half): 0.4156 ± 0.3163
  Predicted T_half: 2.60 days (95% CI: 1.26 - 5.39 days)
  Uncertainty: ± 2.07 days

Molecule 2: CCN
  Predicted log10(T_half): 0.0513 ± 0.3163
  Predicted T_half: 1.13 days (95% CI: 0.54 - 2.33 days)
  Uncertainty: ± 0.89 days

Molecule 3: CCC
  Predicted log10(T_half): 0.2482 ± 0.3163
  Predicted T_half: 1.77 days (95% CI: 0.85 - 3.67 days)
  Uncertainty: ± 1.41 days

Note: Uncertainty based on 5-fold CV RMSE = 0.3163 (log10 space)



/Users/a/dev/WP3biodegradability/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but SVR was fitted with feature names
  warnings.warn(


In [6]:
# Test Option 2: Pass DataFrame directly (with feature names) instead of .values
# This might trigger a different warning or error about feature name matching
import numpy as np

print("Testing Option 2: Passing DataFrame directly to predict()...")
print("-" * 80)

try:
    # Pass the DataFrame directly instead of .values
    predictions_log10_v2 = svr_model.predict(scaled_features)

    # Convert from log10 to original scale (days)
    predictions_days_v2 = 10**predictions_log10_v2

    print("✓ Prediction successful with DataFrame input!")
    print(f"\nPredictions match original method: {np.allclose(predictions_log10, predictions_log10_v2)}")

    # Display results
    print("\nInference Results (DataFrame method):")
    for i, (smiles, pred_days) in enumerate(zip(df["Canonical_smiles"], predictions_days_v2)):
        print(f"Molecule {i + 1}: {smiles} -> {pred_days:.2f} days")

except Exception as e:
    print(f"✗ Error occurred: {type(e).__name__}")
    print(f"  Message: {str(e)}")

Testing Option 2: Passing DataFrame directly to predict()...
--------------------------------------------------------------------------------
✓ Prediction successful with DataFrame input!

Predictions match original method: True

Inference Results (DataFrame method):
Molecule 1: CCO -> 2.60 days
Molecule 2: CCN -> 1.13 days
Molecule 3: CCC -> 1.77 days
